In [1]:
import sys
sys.path.append("../src")
from storage import load_database, DATA_DIR
from queries import count_by_type , history_for_procedure, linked_partner, linked_pairs

In [2]:
conn = load_database(data_dir=DATA_DIR / 'generated')
for t in ['Frequencies', 'Assets', 'Procedures', 'Service_History']:
    print(t, conn.execute(f'SELECT COUNT(*) FROM {t}').fetchone()[0])

Frequencies 4
Assets 40
Procedures 160
Service_History 543


In [3]:
cur = conn.cursor()

count_by_type prototype:

In [4]:
rows = cur.execute("SELECT type, COUNT(*) FROM Assets GROUP BY type ORDER BY COUNT(*) DESC, type").fetchall()
for asset_type, count in rows:
    print(f'{asset_type}: {count}')

generator: 9
motor: 8
pump: 8
transformer: 8
boiler: 4
hvac: 3


In [5]:
rows = count_by_type(conn)
for asset_type, count in rows:
    print(f'{asset_type}: {count}')

generator: 9
motor: 8
pump: 8
transformer: 8
boiler: 4
hvac: 3


In [6]:
assert(rows == count_by_type(conn))

history_for_procedure prototype:

In [7]:
proc_id = 'PMP-002-P1'

sql = '''
    SELECT record_id, date_completed, notes FROM Service_History
    WHERE procedure_id = ?
    ORDER BY date_completed
    '''
rows = cur.execute(sql, (proc_id,)).fetchall()
for record_id, date_completed, notes in rows:
    print(f'{record_id}, {date_completed}, {notes}')

SH-0016, 2026-08-01, None
SH-0017, 2026-08-08, None
SH-0018, 2026-08-17, None
SH-0019, 2026-08-24, None
SH-0020, 2026-08-30, None
SH-0021, 2026-09-05, None
SH-0022, 2026-09-13, None
SH-0023, 2026-09-23, None
SH-0024, 2026-09-26, None
SH-0025, 2026-10-03, None
SH-0026, 2026-10-13, None
SH-0027, 2026-10-17, None
SH-0028, 2026-10-28, None


In [8]:
rows = history_for_procedure(conn ,'PMP-002-P1')
for record_id, date_completed, notes in rows:
    print(f'{record_id}, {date_completed}, {notes}')

SH-0016, 2026-08-01, None
SH-0017, 2026-08-08, None
SH-0018, 2026-08-17, None
SH-0019, 2026-08-24, None
SH-0020, 2026-08-30, None
SH-0021, 2026-09-05, None
SH-0022, 2026-09-13, None
SH-0023, 2026-09-23, None
SH-0024, 2026-09-26, None
SH-0025, 2026-10-03, None
SH-0026, 2026-10-13, None
SH-0027, 2026-10-17, None
SH-0028, 2026-10-28, None


In [9]:
assert(rows == history_for_procedure(conn ,'PMP-002-P1'))

linked_partner prototype:

In [10]:
asset_id = 'PMP-001'

sql = '''
    SELECT a.asset_id, a.type, b.asset_id, b.type 
    FROM Assets as a
    JOIN Assets as b
    ON a.linked_asset_id = b.asset_id
    WHERE a.asset_id = ?
    ORDER BY a.asset_id
    '''
rows = cur.execute(sql, (asset_id,)).fetchall()
for a_id, a_type, b_id, b_type in rows:
    print(f'{a_id}, {a_type}, {b_id}, {b_type}')

PMP-001, pump, MTR-001, motor


In [11]:
rows = linked_partner(conn, 'PMP-001')
for a_id, a_type, b_id, b_type in rows:
    print(f'{a_id}, {a_type}, {b_id}, {b_type}')

PMP-001, pump, MTR-001, motor


In [12]:
assert(rows == linked_partner(conn ,'PMP-001'))

linked_pairs prototype:

In [13]:
asset_id = 'PMP-001'

sql = '''
    SELECT a.asset_id, a.type, b.asset_id, b.type 
    FROM Assets as a
    JOIN Assets as b
    ON a.linked_asset_id = b.asset_id
    WHERE a.asset_id < b.asset_id
    ORDER BY a.asset_id
    '''
rows = cur.execute(sql).fetchall()
for a_id, a_type, b_id, b_type in rows:
    print(f'{a_id}, {a_type}, {b_id}, {b_type}')

MTR-001, motor, PMP-001, pump
MTR-002, motor, PMP-002, pump
MTR-003, motor, PMP-003, pump
MTR-004, motor, PMP-004, pump
MTR-005, motor, PMP-005, pump
MTR-006, motor, PMP-006, pump
MTR-007, motor, PMP-007, pump
MTR-008, motor, PMP-008, pump


In [14]:
rows = linked_pairs(conn)
for a_id, a_type, b_id, b_type in rows:
    print(f'{a_id}, {a_type}, {b_id}, {b_type}')

MTR-001, motor, PMP-001, pump
MTR-002, motor, PMP-002, pump
MTR-003, motor, PMP-003, pump
MTR-004, motor, PMP-004, pump
MTR-005, motor, PMP-005, pump
MTR-006, motor, PMP-006, pump
MTR-007, motor, PMP-007, pump
MTR-008, motor, PMP-008, pump


In [15]:
assert(rows == linked_pairs(conn))

In [16]:
conn.close()